In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "flowmap_legacy").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repository root containing flowmap_legacy")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from flowmap_legacy.evaluation import evaluate_embedding_method


def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time


path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv",
}


In [ ]:
np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.3, 5
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
import dynamo as dyn
import anndata
from matplotlib import colorbar

# Move colorbar to the bottom of each subplot
for cax in fig.axes:
    if isinstance(cax, colorbar.Colorbar):
        cax.ax.set_position([cax.ax.get_position().x0, 0.02, cax.ax.get_position().width, 0.02])
        cax.ax.xaxis.set_ticks_position('bottom')
        cax.ax.xaxis.set_label_position('bottom')


np.random.seed(42)

# ═══════════════ 2. PLOT + SCORE IN ONE LOOP ══════════════════════════════
fig, axes = plt.subplots(1, 8, figsize=(32, 4), constrained_layout=True)
axes = axes.flatten()
records = []

for ax, (name, result) in zip(axes, simulation_results.items()):
    X, V, time = result["X"], result["V"], result["true_time"]
    X_gt, V_gt = result["X_gt"], result["V_gt"]

    # ---- Dynamo pipeline --------------------------------------------------
    adata = anndata.AnnData(X)
    adata.var_names = [f"dim{j}" for j in range(X.shape[1])]
    adata.obs["time"] = time
    adata.layers["X_raw"], adata.layers["V_raw"] = X, V

    dyn.tl.reduceDimension(
        adata,
        X_data=X,
        reduction_method="umap",
        n_neighbors=30,
        min_dist=0.3,
        enforce=True
    )

    dyn.tl.cell_velocities(
        adata,
        ekey="X_raw", vkey="V_raw",
        X=X, V=V,
        X_embedding=adata.obsm["X_umap"],
        basis="umap",
        transition_genes=list(adata.var_names),
        method="pearson",
        enforce=True
    )

    # ---- plotting ---------------------------------------------------------
    dyn.pl.streamline_plot(
        adata,
        basis="umap",
        color="time",
        ax=ax,
        show_legend=False,
        show_colorbar=False,
        show_arrowed_spines=False,
        show_axes=False,
        title=None,
        density=0.2,
        arrowsize=2.0,
        linewidth=2.0,
        streamline_alpha=1.0,
        xy_grid_nums=[50, 50],
        save_show_or_return="return"
    )
    
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(None)

    # ---- scoring ----------------------------------------------------------
    X_emb = adata.obsm["X_umap"]
    V_emb = adata.obsm["velocity_umap"]          # raw projected velocities
    scores = evaluate_embedding_method(X_gt, X_emb, V_gt, V_emb, k=30)
    scores["dataset"] = name
    records.append(scores)

# ═══════════════ 3. CLEAN FIGURE (remove stray colorbars) ═════════════════
for obj in fig.axes:
    if isinstance(obj, colorbar.Colorbar):
        obj.remove()
        
fig_dir = "./figures/simulation"
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "dynamo_embedding_streams.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path}")
plt.show()

# ═══════════════ 4. SAVE SCORE TABLE ══════════════════════════════════════
scores_df = pd.DataFrame(records).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
scores_df.to_csv("./data/8_vf_collection/dynamo.csv")

print("\nSaved Dynamo score table to ./data/8_vf_collection/dynamo.csv\n")
print(scores_df.round(4))